# 分布式训练：工业界的标准工具链

> 「完成第一次预训练与微调」那节的迷你 Trainer，一个模型、一个 optimizer、一个 for 循环，单卡跑得很好。可它搬到 7B 真模型上，第一行 forward 都执行不到：AdamW 全量训练的固定开销就有 112 GB，比一张 A100 整卡显存还大。
>
> 工业界的做法不是手写分布式通信，而是给单卡脚本套一层启动器。代码几乎不动，几张卡怎么分工、显存怎么切、要不要往 CPU 卸载，全由启动时的配置说了算。
>
> 这一节先把 112 GB 这笔账算明白，看懂 ZeRO 在切什么；再往上看从零预训练的重型武器 Megatron；最后回到大多数人的日常，亲手写一份完整的多卡训练脚本，模型、数据、训练循环、checkpoint 一应俱全，写完直接拿走当项目的起点。

先交代边界：多卡之间怎么传数据（all-reduce 的 ring 算法那套底层机制）放在附录《all_reduce 与集合通信》，TP / PP 的手算放在附录《大模型的五种并行切分》。聚焦你真实项目里天天会碰到的三样东西：

- **Megatron-LM**：从零预训练百亿级以上模型的重型武器；
- **ZeRO 参数**：显存不够时真正会去调的那几个配置项（stage、offload、overlap_comm……）；
- **Accelerate**：HuggingFace 的统一启动层，一份训练脚本切换 DDP / FSDP / DeepSpeed。

一条主线贯穿始终：**从零预训练大模型用 Megatron，微调用 HF 生态（Accelerate + ZeRO / FSDP）**。

In [ ]:
# 本章所有 import 集中放在第一个 code cell
import json
import os
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from accelerate import Accelerator

torch.manual_seed(42)
print("torch:", torch.__version__)

## 1. 先算一笔账：单卡为什么不够

先回答「为什么要分布式」。AdamW + 混合精度训练下，每个参数要占 16 bytes：FP16 参数 2 bytes、FP16 梯度 2 bytes、FP32 master 权重 4 bytes、AdamW 一阶矩 4 bytes、二阶矩 4 bytes。这部分是和序列长度无关的固定开销，7B 参数就是 112 GB，还没算激活值。

多卡最朴素的用法是数据并行（Distributed Data Parallel，DDP）：每张卡持有一份完整模型副本，把数据切成 N 份分给 N 张卡，各自前向反向，再用 all-reduce 把梯度平均。它的吞吐接近线性增长，但有个致命问题——**显存一点没省**：8 张卡总共 896 GB 显存里，有 7/8 是一模一样的冗余副本。

In [ ]:
# === 单卡显存账单：7B 模型 ===
P = 7e9  # 7B 参数

param_fp16   = 2 * P   # FP16 参数
grad_fp16    = 2 * P   # FP16 梯度
master_fp32  = 4 * P   # FP32 master 权重
adam_m       = 4 * P   # AdamW 一阶矩
adam_v       = 4 * P   # AdamW 二阶矩

fixed_bytes = param_fp16 + grad_fp16 + master_fp32 + adam_m + adam_v
fixed_gb = fixed_bytes / 1e9

print(f"模型参数 (FP16):    {param_fp16 / 1e9:.1f} GB")
print(f"梯度 (FP16):        {grad_fp16 / 1e9:.1f} GB")
print(f"master 权重 (FP32): {master_fp32 / 1e9:.1f} GB")
print(f"AdamW m (FP32):     {adam_m / 1e9:.1f} GB")
print(f"AdamW v (FP32):     {adam_v / 1e9:.1f} GB")
print(f"固定开销小计:       {fixed_gb:.0f} GB (= 16 × P bytes)")
print()
print("关键观察：16P = 2P 参数 + 2P 梯度 + 12P 优化器状态。")
print("DDP 下这三块在每张卡上都是完整副本——冗余就在这里。")

## 2. ZeRO：把冗余切到多卡

ZeRO（Zero Redundancy Optimizer）是微软 2020 年提出的方法，也是今天所有主流训练框架显存优化的基石。思路一句话：既然每张卡上的 16P bytes 都是冗余副本，那就按三块逐层切到 N 张卡上，切完的卡互相配合，逻辑上仍然等价于一个完整的大模型。

三个 Stage 递进，每升一级多切一块：

- **Stage 1**：切优化器状态（12P → 12P/N），每卡只更新自己那 1/N 参数的 optimizer；
- **Stage 2**：再切梯度（2P → 2P/N），反向时用 reduce-scatter 让每卡只留自己那片梯度；
- **Stage 3**：再切参数（2P → 2P/N），前向反向用到哪层就临时 all-gather 哪层，用完即丢。

In [ ]:
# === ZeRO 三 Stage 显存手算：7B 模型 × 8 卡 ===
P = 7e9
N = 8

print(f"{'方案':<16}{'参数':>8}{'梯度':>8}{'优化器':>8}{'单卡合计':>10}{'相对 DDP':>10}")
print("-" * 64)
configs = [
    ("DDP",          2 * P,       2 * P,       12 * P),
    ("ZeRO Stage 1", 2 * P,       2 * P,       12 * P / N),
    ("ZeRO Stage 2", 2 * P,       2 * P / N,   12 * P / N),
    ("ZeRO Stage 3", 2 * P / N,   2 * P / N,   12 * P / N),
]
ddp_bytes = 16 * P
for name, p, g, o in configs:
    total = p + g + o
    print(f"{name:<16}{p/1e9:>7.1f} {g/1e9:>7.1f} {o/1e9:>7.1f} "
          f"{total/1e9:>8.1f} GB {total/ddp_bytes*100:>8.1f}%")
print()
print("关键观察：从 DDP 到 Stage 3，单卡显存 112 GB → 14 GB，压缩到 1/8。")
print("代价是通信量递增：Stage 越高，前向反向要传递的碎片越多。")
print()
print("工业经验：Stage 2 是性价比最高的档位（通信开销 ≈ DDP）；")
print("参数实在放不下才上 Stage 3（对节点间带宽更敏感）。")

## 3. 预训练大模型：Megatron 的 3D 并行

ZeRO 的两个主流工业实现是 DeepSpeed 和 PyTorch FSDP，但不管哪个，本质都是数据并行的变体。模型到几百亿、集群到几千卡时会撞上天花板：ZeRO-3 每层前向都要 all-gather 参数，跨节点通信量随规模爆炸。

从零预训练大模型的工业标准是 Megatron-LM 的 **3D 并行**：

- **Tensor Parallelism（TP，张量并行）**：把每一层的矩阵乘按维度切到多卡。通信是每层都有的 all-reduce，对延迟极敏感，**只放节点内**吃 NVLink；
- **Pipeline Parallelism（PP，流水线并行）**：把模型按层切成若干段，不同段在不同节点上接力。通信量小（只传边界激活），**适合跨节点**容忍高延迟；
- **Data Parallelism（DP）**：最外层再复制几份，用 ZeRO 切冗余。

记住一条铁律就够：**TP 锁节点内，PP 跨节点，剩下的卡全给 DP**。

Megatron 的定位和 DeepSpeed / FSDP 完全不同：它们是「给数据并行训练加显存优化」，Megatron 是「一整套从零预训练的框架」，连 GPT 模型定义、数据加载、loss、checkpoint 都替你写好了。它最常见的几个参数感受一下：

| 参数 | 含义 |
|:---|:---|
| `--tensor-model-parallel-size 8` | TP 度 = 8：每层矩阵乘切 8 份，锁在节点内 8 卡 |
| `--pipeline-model-parallel-size 16` | PP 度 = 16：模型按层切 16 段，跨节点接力 |
| `--global-batch-size 1024` | 一个完整 step 的样本数，框架自动推算梯度累积步数 |
| `--sequence-parallel` | 序列维也切分，和 TP 配合省激活显存 |
| `--recompute-activations` | 重算激活（梯度检查点），预训练省显存标配 |

| 场景 | 工具 |
|:---|:---|
| 从零预训练 70B+、几千卡 | Megatron 系（3D 并行） |
| 从零预训练 7B~13B | Accelerate + FSDP 也够用 |
| 微调、几百卡以内 | Accelerate + ZeRO / FSDP |

（TP / PP 的内部原理和手算放在附录《大模型的五种并行切分》；近年 PyTorch 官方的 torchtitan 走 FSDP + TP 轻量路线，适合中等规模。）

## 4. 显存不够时动哪几个旋钮

显存 OOM 是分布式训练的日常。工业界的排查顺序从代价小到代价大：

1. 先降 micro-batch、加梯度累积（零成本，用时间换空间）；
2. ZeRO Stage 2 → Stage 3；
3. `offload_optimizer`：把优化器状态卸到 CPU 内存（省最多，但 CPU↔GPU 搬运有开销）；
4. `offload_param`：参数也卸到 CPU（只有 Stage 3 支持）；
5. 实在不行再考虑 NVMe 硬盘 offload（ZeRO-Infinity，慢但能救急）。

这些旋钮在 DeepSpeed 的 JSON 配置里长这样（下一节的 Accelerate 里通过 `--ds_config_file` 传同一份东西）：

In [ ]:
# === DeepSpeed ZeRO 配置：工业项目里最常改的几个字段 ===
deepspeed_config = {
    "train_micro_batch_size_per_gpu": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,                    # 1 切优化器状态 / 2 再切梯度 / 3 再切参数
        "offload_optimizer": {         # 优化器状态卸到 CPU 内存
            "device": "cpu",
            "pin_memory": True,        # 锁页内存，CPU→GPU 拷贝更快
        },
        "offload_param": {             # 参数也卸到 CPU（仅 stage 3 生效）
            "device": "none",
        },
        "overlap_comm": True,          # 通信和计算重叠，藏掉一部分通信延迟
        "contiguous_gradients": True,  # 梯度存成连续内存块，减少通信碎片
        "reduce_bucket_size": 5e8,     # 梯度分桶大小（bytes），大桶通信高效但耗显存
    },
}

print("ds_config.json:")
print(json.dumps(deepspeed_config, indent=2))
print()
print("启动：accelerate launch --use_deepspeed --ds_config_file ds_config.json train.py")

真正经常动的只有 stage 和两个 offload，其余保持默认就好：

| 参数 | 干什么用 | 什么时候动它 |
|:---|:---|:---|
| `stage`（1/2/3） | 切优化器状态 / 梯度 / 参数 | 显存不够就升级，常用 2 和 3 |
| `offload_optimizer` | 优化器状态卸到 CPU | Stage 3 还不够时，拿速度换显存 |
| `offload_param` | 参数也卸到 CPU | 模型大到 GPU 放不下，仅 Stage 3 |
| `overlap_comm` | 通信与计算重叠 | 默认开；多卡通信慢时确认它开着 |
| `contiguous_gradients` | 梯度连续存储 | 默认开，一般不用管 |
| `reduce_bucket_size` | 梯度分桶大小（bytes） | 通信是瓶颈时可调大，代价是显存 |

如果后端选 FSDP 而不是 DeepSpeed，概念几乎一一映射：`FULL_SHARD` = ZeRO-3，`SHARD_GRAD_OP` = ZeRO-2。两套怎么选？能力高度重合，算工程口味：DeepSpeed 配置驱动、offload 生态更全；FSDP 是 PyTorch 亲儿子、跟进新硬件最快。下一节的 Accelerate 把两者都包起来，代码不用改。

## 5. 动手写：一份能直接拿走的多卡训练脚本

重型武器看完了，回到大多数人的日常：微调，几百卡以内。接下来动真格，把「完成第一次预训练与微调」那节的迷你 Trainer 升级成多卡版。

直接用 PyTorch 的话，换一个后端就要改一次代码：DDP 要 `torchrun` 启动加 DistributedSampler 包 DataLoader，FSDP 要用包装器套模型，DeepSpeed 要初始化 engine 再配 JSON。Accelerate 把这些差异全收进一个 `Accelerator` 对象：它**不是另一套分布式算法**，DDP 模式下底层是 torch.distributed，FSDP 模式下是 PyTorch FSDP，DeepSpeed 模式下就是 DeepSpeed，你只面对同一套 API。

升级分五步：模型与数据、Accelerator 初始化、训练循环、checkpoint、启动命令。每一步的产物都真实可运行。

### 5.1 模型与数据：先写一个普通的单卡脚本

模型还是 embedding 接 lm_head 的 TinyCausalLM，数据还是几条带固定模式的 token 序列。麻雀虽小，接口和真模型一模一样：Dataset 出单条样本，DataLoader 拼 batch，模型吃 `input_ids` 和 `labels` 返回 loss。先把这部分写成和单卡完全相同的样子——多卡改造一行都不用碰它。

In [ ]:
# === 模型与数据：这部分和单卡脚本完全相同 ===
class TinyCausalLM(nn.Module):
    """极小的 Causal LM：embedding 接 lm_head，用来看训练行为。"""

    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, labels=None):
        """
        输入 token ids 输出 logits；给了 labels 就顺便算 loss。

        input_ids: [batch, seq_len]，去掉结尾 token 的样本
        labels:    [batch, seq_len]，左移一位的下一个 token
        """
        logits = self.lm_head(self.embedding(input_ids))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return {"loss": loss, "logits": logits}


class ToyTextDataset(Dataset):
    """每条样本是一小段 token ids：[1] 开头 [2] 结尾，中间是固定模式。"""

    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        ids = self.sequences[index]
        # 输入取前 n-1 个 token，监督信号是后 n-1 个（预测下一个 token）
        return {
            "input_ids": torch.tensor(ids[:-1]),
            "labels": torch.tensor(ids[1:]),
        }


def simple_collate(features):
    """把一批样本 dict 拼成整批 tensor，模拟工业 collator 的职责。"""
    return {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
    }


# 两种交替出现的固定模式，模型能学出「1 后面是 3」「3 后面是 4」这类规律
sequences = [
    [1, 3, 4, 5, 6, 2] if i % 2 == 0 else [1, 3, 4, 6, 5, 2]
    for i in range(16)
]

dataset = ToyTextDataset(sequences)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=simple_collate)

model = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

batch = next(iter(dataloader))
print("一个 batch 的形状：", {k: tuple(v.shape) for k, v in batch.items()})
print()
print("关键观察：到这里为止没有一行分布式代码，它就是普通的单卡脚本。")

### 5.2 Accelerator：一次 prepare，接管所有后端差异

多卡改造从创建 `Accelerator` 开始。它有两个职责：吃进训练三件套（model、optimizer、dataloader），按当前后端把它们包装好；再提供几个取代原生写法的 API。`mixed_precision` 在真卡上配 `bf16`，这里 CPU 演示先关掉：

In [ ]:
# === Accelerator：创建并 prepare，多卡差异从此被接管 ===
accelerator = Accelerator(
    gradient_accumulation_steps=2,  # 2 个 micro-batch 攒够，才算一次完整 step
    mixed_precision="no",           # CPU 演示不开混合精度；真卡上配 "bf16"
)

model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

print("distributed_type:", accelerator.distributed_type)
print("num_processes:   ", accelerator.num_processes)
print("process_index:   ", accelerator.process_index)
print("is_main_process: ", accelerator.is_main_process)
print()
print("关键观察：现在 num_processes = 1，看不出动静；换成")
print("accelerate launch --num_processes 8 启动后，每个进程拿到不同的")
print("process_index 和不同的数据分片，而这份代码一个字都不用改。")

### 5.3 训练循环：相比单卡只改三个地方

对照「完成第一次预训练与微调」那节的迷你 loop，改动一共三处，每处都值得记一笔：

- `with accelerator.accumulate(model)`：梯度累积的管理员。没攒够 2 个 micro-batch，它会拦住 `optimizer.step()` 不放行，攒够了才真正更新一次参数；
- `accelerator.backward(loss)`：取代 `loss.backward()`。DDP 模式下它顺手完成跨卡梯度同步，DeepSpeed ZeRO 模式下它负责梯度的切与聚；
- 日志包上 `is_main_process`：8 个进程只有 0 号该打印，否则每行日志重复 8 份。

In [ ]:
# === 完整训练循环：这份 loop 可以原封不动搬进 train.py ===
num_epochs = 8

for epoch in range(num_epochs):
    total_loss, num_steps = 0.0, 0

    for batch in dataloader:
        with accelerator.accumulate(model):
            outputs = model(batch["input_ids"], labels=batch["labels"])
            accelerator.backward(outputs["loss"])
            optimizer.step()
            optimizer.zero_grad()

        if accelerator.is_main_process:
            total_loss += outputs["loss"].item()
            num_steps += 1

    if accelerator.is_main_process and (epoch + 1) % 2 == 0:
        print(f"epoch {epoch + 1:02d} | train_loss = {total_loss / num_steps:.4f}")

print()
print("关键观察：loss 在稳定下降，多卡化没有改变训练行为。")
print("同一份 loop，单进程是单卡训练，8 进程启动就是 8 卡数据并行。")

### 5.4 有效 batch：三个数的乘积

`gradient_accumulation_steps=2` 在刚才的循环里做了什么？它和另外两个数一起，决定每次参数更新「吃」了多少样本：

$$\text{有效 batch} = \text{micro-batch} \times \text{GPU 数} \times \text{累积步数}$$

刚才的演示是 4 × 1 × 2 = 8。放到真项目里，micro-batch 4 × 8 卡 × 累积 4 步 = 有效 batch 128：单卡一次只装 4 个样本的激活，但优化 step 用的是 128 个样本的梯度。显存按 micro-batch 算，训练效果按有效 batch 算，这就是「用时间换空间」。

另外两个高频启动参数也在这说清。**`num_processes`** 是 GPU 总数，8 卡就是 8 个进程，每个进程跑同一份脚本、处理不同的数据分片。**`mixed_precision`** 工业界默认 `bf16`：A100/H100 原生支持，数值范围和 FP32 一样大，不需要 FP16 那套 loss scale，训练更稳（细节见附录《混合精度训练与 loss scaling》）。

### 5.5 checkpoint：中断续训不是可选项

真实训练动辄几天几周，机器重启、抢占、断电都会打断训练。Accelerate 的配套 API 是一对：`save_state` 把模型、optimizer、dataloader 进度整体落盘；`load_state` 在任何新进程里原样恢复。调用时不用判断主进程，每个进程都调它，多进程状态一致性由它自己保证。

In [ ]:
# === checkpoint 演示：存档 → 模拟新进程恢复 → 验证权重一致 ===
ckpt_dir = "_ckpt_demo"

accelerator.save_state(ckpt_dir)
print("save_state 落盘的文件：")
for name in sorted(os.listdir(ckpt_dir)):
    print(" ", name)

# 模拟「换台机器从存档恢复」：新进程 = 新的 Accelerator + 没训练过的新模型
accelerator_restored = Accelerator(
    gradient_accumulation_steps=2,
    mixed_precision="no",
)
model_restored = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer_restored = torch.optim.AdamW(model_restored.parameters(), lr=0.05)
model_restored, optimizer_restored = accelerator_restored.prepare(
    model_restored, optimizer_restored
)
accelerator_restored.load_state(ckpt_dir)

w_trained = accelerator.unwrap_model(model).embedding.weight
w_restored = accelerator_restored.unwrap_model(model_restored).embedding.weight
assert torch.allclose(w_trained, w_restored), "恢复后的权重应和存档时一致"

print()
print("关键观察：load_state 后权重逐位一致，")
print("optimizer 状态（Adam 的一阶矩二阶矩）也一起恢复了。")
shutil.rmtree(ckpt_dir)  # 演示完清掉，不留垃圾文件

### 5.6 启动：代码不动，命令动

脚本到此写完。怎么跑上多卡？答案是不改代码，换命令。注意下面的训练脚本始终是同一份 `train.py`，就是前面几个小节内容的合集：

In [ ]:
# === 启动命令：单卡、多卡、DeepSpeed，训练脚本都是同一份 ===
print("单进程（等价于 python train.py）：")
print("  $ accelerate launch train.py")
print()
print("单机 8 卡：")
print("  $ accelerate launch --num_processes 8 --multi_gpu train.py")
print()
print("DeepSpeed ZeRO-3 后端（搭配上一节的 ds_config.json）：")
print("  $ accelerate launch --use_deepspeed --ds_config_file ds_config.json \\")
print("      --num_processes 8 train.py")
print()
print("不想记参数，就先跑一次 accelerate config（交互式问答）生成配置文件，")
print("之后永远 accelerate launch --config_file xxx.yaml train.py。")
print()
print("关键观察：DDP 换 FSDP 换 DeepSpeed，改的只有启动命令，")
print("5.1~5.5 节的代码一行不动。")

## 6. 微调的标配装备

最后补几个和分布式训练搭着用的高频开关，工业项目里几乎每个微调脚本都有它们的影子：

- **梯度累积**：显存装不下大 batch 时用时间换空间；
- **梯度检查点**（gradient checkpointing）：反向时不保存中间激活，用到时重算。激活显存省 60% 以上，整体慢约 30%；
- **BF16 混合精度**：参数和激活用 BF16 存，计算快一倍、显存省一半；
- **FlashAttention-2**：注意力计算不物化完整 attention 矩阵，长序列场景显存和速度双收益（原理在附录《FlashAttention 的分块计算》）；
- **8-bit 优化器**：AdamW 状态从 12P 压到 3P bytes，效果几乎无损。

HuggingFace 生态里的写法：

| 装备 | 解决什么问题 | 典型参数写法 |
|:---|:---|:---|
| 梯度累积 | batch 大显存装不下 | `gradient_accumulation_steps=8` |
| 梯度检查点 | 激活值吃显存 | `gradient_checkpointing=True` |
| BF16 混合精度 | 算得快、显存省 | `bf16=True` |
| FlashAttention-2 | 长序列注意力慢且占显存 | `attn_implementation='flash_attention_2'` |
| 8-bit 优化器 | 优化器状态 12P 太大 | `optim='adamw_bnb_8bit'` |

典型组合：7B 模型单卡 LoRA 微调 = bf16 + 梯度累积 + FlashAttention-2；70B 模型多卡全量微调 = bf16 + ZeRO-3 + offload + 梯度检查点。

## 小结

- [ ] 7B 全量训练固定开销 ≈ 112 GB = 16 bytes × 参数量，DDP 多卡一点不省
- [ ] ZeRO Stage 1/2/3 分别切优化器状态、梯度、参数，越切越省、通信越多
- [ ] Megatron 用 3D 并行从零预训练大模型：TP 锁节点内，PP 跨节点，DP 填满剩余
- [ ] OOM 排查顺序：梯度累积 → stage 2 → stage 3 → offload optimizer → offload param
- [ ] 多卡脚本 = 单卡脚本 + 四处改动：Accelerator、prepare、accumulate/backward、is_main_process
- [ ] 有效 batch = micro-batch × GPU 数 × 梯度累积步数
- [ ] checkpoint 用 save_state / load_state，模型和 optimizer 状态一起恢复
- [ ] 工业分工：从零预训练大模型用 Megatron 系，微调用 Accelerate + ZeRO/FSDP

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：算 ZeRO Stage 2 在 4 卡下的单卡显存**

7B 模型，AdamW 训练，4 张卡。ZeRO Stage 2 下每张卡的固定显存是多少 GB？

小提示：Stage 2 切梯度和优化器状态（14P 切成 4 份），参数仍每卡完整保留（2P）。

In [ ]:
# 作业 1：ZeRO Stage 2 在 4 卡下的单卡显存
P = 7e9
N = 4

# TODO: 计算 Stage 2 的单卡显存（单位 GB）
# 参数完整保留 + (梯度 + 优化器状态) 切 N 份
s2_per_card_gb = (2 * P + 14 * P / N) / 1e9

assert s2_per_card_gb is not None, "请先计算 Stage 2 单卡显存"
expected = (2 * P + 14 * P / N) / 1e9
assert abs(s2_per_card_gb - expected) < 0.1, f"应为 {expected:.1f} GB"
print(f"✅ 作业 1 通过：")
print(f"   Stage 2 + 4 卡 + 7B：单卡 {s2_per_card_gb:.1f} GB")
print(f"   相比 DDP 的 112 GB 省了 {112 - s2_per_card_gb:.1f} GB，通信代价却几乎没涨。")

**作业 2：写一份「显存告急」的 DeepSpeed 配置**

补全下面的 `deepspeed_config`，要求：ZeRO Stage 3、优化器状态 offload 到 CPU、开启通信计算重叠。

小提示：对应 `zero_optimization.stage`、`offload_optimizer.device`、`overlap_comm` 三个字段。

In [ ]:
# 作业 2：写显存告急时的 DeepSpeed 配置
deepspeed_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu"},
        "overlap_comm": True,
    },
}

# 验证
zero = deepspeed_config.get("zero_optimization", {})
assert zero.get("stage") == 3, "stage 应为 3"
assert zero.get("offload_optimizer", {}).get("device") == "cpu", "offload_optimizer.device 应为 'cpu'"
assert zero.get("overlap_comm") is True, "overlap_comm 应为 True"

print("✅ 作业 2 通过：")
print(json.dumps(deepspeed_config, indent=2))
print()
print("这份配置 = 显存最紧张时的第一档方案：参数/梯度/优化器全切 + 优化器上 CPU。")

**作业 3：算有效 batch size**

一个预训练任务：micro-batch 2，32 张卡，梯度累积 8 步。一个完整优化 step 用了多少个样本？

小提示：三个数相乘，这就是 5.4 节公式里三个因子的现实版本。

In [ ]:
# 作业 3：算有效 batch size
micro_batch = 2
num_gpus = 32
grad_accum = 8

# TODO: 计算有效 batch size
effective_batch = micro_batch * num_gpus * grad_accum

assert effective_batch is not None, "请先计算有效 batch size"
expected = micro_batch * num_gpus * grad_accum
assert effective_batch == expected, f"应为 {expected}"
print(f"✅ 作业 3 通过：")
print(f"   有效 batch = {micro_batch} × {num_gpus} × {grad_accum} = {effective_batch}")
print(f"   单卡一次只需要装 {micro_batch} 个样本的激活，却达到了 {effective_batch} 的 batch 效果。")

## 参考资料

- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- [HuggingFace Accelerate 文档](https://huggingface.co/docs/accelerate/)
- [DeepSpeed ZeRO 配置项文档](https://www.deepspeed.ai/docs/config-json/)
- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- [NVIDIA Megatron-LM GitHub](https://github.com/NVIDIA/Megatron-LM)